## Interactive audio sample cleaning

TODO:

* The `SpectrogramAnnotator` class silently screws up time scale when not using 32 kHz
* Add a primary_label column to metadata

In [22]:
from pathlib import Path
project_root = str(Path().resolve().parent)

In [23]:
geographic_extents = {'new_zealand': {'min_longitude': 166,
                                        'max_longitude': 179,
                                        'min_latitude': -49,
                                        'max_latitude': -34},
                      'south_america': {'min_longitude': -82,
                                        'max_longitude': -34.0,
                                        'min_latitude': -56.0,
                                        'max_latitude': 13.0}
                      }

### This cell requires manual setup

In [24]:
use_case = {
            'project_root': project_root,
            'source_dataset_path': project_root + '/data/anqa_from_avianz',
            'reviewed_data_destn':project_root + '/data/anqa_reviewed',
            'naming_csv':  project_root + '/data/bird_names/bird_names.csv',  #Requires eBird and CommonName columns, and must include Unknown | unknown
            'audio_folder':  Path(project_root + '/data/anqa_from_avianz/audio'),
            'author': 'Olly',   #Use Author for first annotation, otherwise None
            'reviewer': None,   #Use Reviwer for subsequent corrections/additions
            'map_extents': geographic_extents['new_zealand'],
            'display_width': 16, #Adjust for screen size, 
            }

load_samples = False

In [25]:
class FilePaths:
    def __init__(self, options: dict):
        _project_dir = Path(options['project_root'])
        self.original_dataset = Path(options['source_dataset_path'])
        self.data_folder = _project_dir / 'data'
        self.audio_folder = options.get('audio_folder', self.data_folder)
        self.original_labels = self.original_dataset / 'annotations.parquet'
        self.original_metadata = self.original_dataset / 'metadata.parquet'
        self.out_dataset = Path(options['reviewed_data_destn'])
        self.out_dataset.mkdir(exist_ok=True, parents=True)
        self.out_labels = self.out_dataset / 'annotations.parquet'
        self.out_metadata = self.out_dataset / 'metadata.parquet'
        self.naming_csv = Path(options['naming_csv'])

In [26]:
from __future__ import annotations  #Not needed for Python 3.10+
import matplotlib
import pandas as pd
from IPython.display import display #, HTML
import ipywidgets as widgets
button = widgets.Button(description="Continue")
output = widgets.Output()
import contextily as cx

from anqa.annotation import (MiniBirdNamer, FastMap, AnnotationState,
                             normalize_secondary_labels, create_class_widgets,
                             SpectrogramAnnotator, AnnotationSession,
                             AnnotationControls, load_current_sample)

%matplotlib widget  
print(f'The Matplotlib backend is {matplotlib.get_backend()}')

The Matplotlib backend is widget


### Initialize

In [27]:
paths = FilePaths(use_case)
namer = MiniBirdNamer(paths.naming_csv)
map = FastMap(map_extents=use_case['map_extents'],
              provider =cx.providers.OpenStreetMap.Mapnik) # type: ignore[attr-defined])    #.OpenTopoMap, OpenStreetMap.Mapnik, 
annotation_state = AnnotationState(all_classes=namer.common_names, namer=namer, max_visible=30)

## Check the data tables
Start by loading the metadata dataframe, there should be exactly one row per file

In [28]:
df_meta = pd.read_parquet(paths.original_metadata)
df_meta = df_meta.sort_values(by='filename')  #Ensures all the files from one class folder are presented sequentially
df_meta['secondary_labels'] = df_meta['secondary_labels'].apply(normalize_secondary_labels)
df_meta.head(3)

,filename,collection,secondary_labels,url,latitude,longitude,author,license,recorded_on,reviewed_by,reviewed_on,source_filename,source_start_s,source_end_s,models_used
30,20190831_074504_from_0.flac,NaN,[],NaN,NaN,NaN,Sumudu,NaN,2019-08-31 07:45:04,NaN,NaN,20190831_074504.wav,0.0,60.0,NaN
32,20190831_074504_from_120.flac,NaN,[],NaN,NaN,NaN,Sumudu,NaN,2019-08-31 07:45:04,NaN,NaN,20190831_074504.wav,120.0,180.0,NaN
33,20190831_074504_from_180.flac,NaN,[],NaN,NaN,NaN,Sumudu,NaN,2019-08-31 07:45:04,NaN,NaN,20190831_074504.wav,180.0,240.0,NaN


In [29]:
df_meta['primary_label'] = 'nezkak1'  #just for testing purposes

In [30]:
len(df_meta)

45

In [31]:
df_meta.head(3)

,filename,collection,secondary_labels,url,latitude,longitude,author,license,recorded_on,reviewed_by,reviewed_on,source_filename,source_start_s,source_end_s,models_used,primary_label
30,20190831_074504_from_0.flac,NaN,[],NaN,NaN,NaN,Sumudu,NaN,2019-08-31 07:45:04,NaN,NaN,20190831_074504.wav,0.0,60.0,NaN,nezkak1
32,20190831_074504_from_120.flac,NaN,[],NaN,NaN,NaN,Sumudu,NaN,2019-08-31 07:45:04,NaN,NaN,20190831_074504.wav,120.0,180.0,NaN,nezkak1
33,20190831_074504_from_180.flac,NaN,[],NaN,NaN,NaN,Sumudu,NaN,2019-08-31 07:45:04,NaN,NaN,20190831_074504.wav,180.0,240.0,NaN,nezkak1


In [32]:
df_labels = pd.read_parquet(paths.original_labels)
df_labels.head(3)

,Filename,Start Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),Label,Type,Sex,Score,Delta Time (s),Delta Freq (Hz),Avg Power Density (dB FS/Hz)
0,20190831_083004_from_420.flac,37.6,45.0,1230.0,2365.0,weka1,<NA>,<NA>,NaN,7.4,1135.0,-53.0
1,20190831_181504_from_240.flac,54.5,60.0,997.0,2656.0,weka1,<NA>,<NA>,NaN,5.5,1659.0,-62.1
2,20190831_181504_from_300.flac,0.0,5.1,997.0,2656.0,weka1,<NA>,<NA>,NaN,5.1,1659.0,-61.2


In [33]:
df_naming = pd.read_csv(paths.naming_csv)
df_naming.head(3)

,CommonName,eBird,ScientificName,ExtraName
0,Bellbird,nezbel1,Anthornis melanura,Bellbird
1,Australasian Bittern,ausbit1,Botaurus poiciloptilus,Bittern
2,Bittern,nezbit1,Ixobrychus novaezelandiae,Bittern


## Annotation
* **Every** Bird or Animal Sound to be boxed
* Unknown classes to be labelled *Unknown*
* Calls from the same bird with gaps of greater than 2 seconds should have individual boxes
* Otherwise a single large box should be used
* Spacebar to clear all labels
* Right click to clear the last label
* The t key will attempt to box all patterns matching the contents of the most recent box

In [34]:
create_class_widgets(annotation_state, n_columns=6, fastmap=map, common_to_ebird=namer.common_to_ebird_dict)

annotator = SpectrogramAnnotator(annotation_state,
                                 common_to_ebird=namer.common_to_ebird_dict,
                                 plot_size = (use_case['display_width'],4),  #Adjust for screen size
                                 f_min=20,
                                 f_max=16000,
                                 zoom_window_height=0.4,
                                 zoom_window_width=5,
                                 min_drag_rows=5,
                                 min_drag_time_s=.1,
                                 min_separation = 2,
                                 similarness_threshold=0.5,
                                 min_freq_hz=300)    ####################Not working yet #########################

session = AnnotationSession(df_meta=df_meta,
                            df_labels=df_labels,
                            new_meta_filepath=paths.out_metadata,
                            new_labels_filepath=paths.out_labels,
                            reviewer = use_case['reviewer'],
                            author = use_case['author'])
controls = AnnotationControls()
annotation_widget = load_current_sample(session, annotator, paths, map)
controls.display()
controls.bind(session, annotator, paths, map)

def on_space_key(event):
    if event.key == ' ':
        controls._on_next_clicked()

annotator.fig.canvas.mpl_connect('key_press_event', on_space_key)

20

## Shortcuts

| Action | Result |
|--------| ---------------- |
| **Left mouse click-drag** |Starts box drawing on click, finishes on release |
| **Left mouse click** |Repeats previous box, but centred on the pointer|
| **Right mouse click** | Moves the zoom box and restarts the playback at that point|
| **u** | Undoes the last box |
| **d** | Deletes all boxes (including originals) |
| **t** | Tries to box any identical patterns from the last un the same frequency limits |
| **b** | Tries to mark calls based on power peaks.  Not recommended, needs improvement |
| **g** | Places a grid of 10 second spacing |
| **g again** | A vertical and horizontal grid |
| **g again** | No grid |

### Progress Checks

In [35]:
session.summary()

{'total_files': 45,
 'finished_files_in_new_meta': 16,
 'total_annotations': 100,
 'done_in_current_session': np.int64(16),
 'pending_in_current_session': np.int64(29),
 'total_minus_pending': np.int64(16)}

In [36]:
marked_times = annotator.get_boxes()
marked_times[-1:]

[]

In [37]:
if paths.out_metadata.exists():
    output_metadata = pd.read_parquet(paths.out_metadata)
    display(output_metadata.tail())

,filename,collection,primary_label,secondary_labels,url,latitude,longitude,author,license,recorded_on,reviewed_by,reviewed_on,source_filename,source_start_s,source_end_s,models_used
11,20190831_074504_from_660.flac,<NA>,nezkak1,[],<NA>,NaN,NaN,Sumudu,<NA>,2019-08-31 07:45:04,<NA>,NaN,20190831_074504.wav,660.0,720.0,<NA>
12,20190831_074504_from_720.flac,<NA>,nezkak1,[],<NA>,NaN,NaN,Sumudu,<NA>,2019-08-31 07:45:04,<NA>,NaN,20190831_074504.wav,720.0,780.0,<NA>
13,20190831_074504_from_780.flac,<NA>,nezkak1,[],<NA>,NaN,NaN,Sumudu,<NA>,2019-08-31 07:45:04,<NA>,NaN,20190831_074504.wav,780.0,840.0,<NA>
14,20190831_074504_from_840.flac,<NA>,nezkak1,[],<NA>,NaN,NaN,Sumudu,<NA>,2019-08-31 07:45:04,<NA>,NaN,20190831_074504.wav,840.0,900.0,<NA>
15,20190831_083004_from_0.flac,<NA>,nezkak1,[],<NA>,NaN,NaN,Olly,<NA>,2019-08-31 08:30:04,<NA>,2026-07-01,20190831_083004.wav,0.0,60.0,<NA>


In [38]:
if paths.out_metadata.exists():
    display(output_metadata.shape)

(16, 16)

In [39]:
if paths.out_labels.exists():
    output_labeldata = pd.read_parquet(paths.out_labels)
    display(output_labeldata.tail(5))

,Filename,Start Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),Label,Type,Sex,Score,Life Stage,Indv ID,Delta Time (s),Delta Freq (Hz),Avg Power Density (dB FS/Hz)
88,20190831_074504_from_60.flac,55.632,58.608,3921.875,6640.625,tomtit1,Begging,Male,NaN,Juvenile,None,3.0,2719.0,-59.9
89,20190831_074504_from_60.flac,38.080,41.600,1937.500,3921.875,tomtit1,Begging,Male,0.7,Juvenile,None,3.5,1984.0,-44.0
90,20190831_074504_from_60.flac,11.456,16.608,1718.750,3687.500,tomtit1,Begging,Male,0.7,Juvenile,None,5.2,1969.0,-47.8
91,20190831_083004_from_0.flac,47.968,55.424,1718.750,5562.500,nezkak1,NaN,NaN,NaN,NaN,None,7.5,3844.0,-39.1
92,20190831_083004_from_0.flac,51.024,57.248,500.000,1140.625,nezkak1,NaN,NaN,NaN,NaN,None,6.2,641.0,-22.7
